# Module 3 — Topic 2: Identifying Patterns & Outliers
## Live Demo Notebook

**Scenario:** You've been given a salary dataset of 500 Nigerian professionals across 10 sectors.
Your task: identify any suspicious salary values, understand the distribution, and recommend the right summary statistic to report.

---

**My EDA questions:**
1. Are there any salary values that look unusually high or low?
2. Is the salary distribution skewed — and if so, which way?
3. Does salary increase with years of experience?

---

## Part 1 — Load and Inspect

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('nigerian_salaries.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Spot the signal first: mean vs median
mean_sal   = df['salary_monthly'].mean()
median_sal = df['salary_monthly'].median()

print(f'Mean salary:   NGN {mean_sal:,.0f}')
print(f'Median salary: NGN {median_sal:,.0f}')
print(f'Difference:    NGN {mean_sal - median_sal:,.0f}')
print()
print('>>> Large gap between mean and median — likely outliers pulling the mean up.')

---
## Part 2 — IQR Outlier Detection

In [ ]:
# Step 1: Calculate quartiles and IQR
Q1  = df['salary_monthly'].quantile(0.25)
Q3  = df['salary_monthly'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'Q1  (25th percentile): NGN {Q1:,.0f}')
print(f'Q3  (75th percentile): NGN {Q3:,.0f}')
print(f'IQR:                   NGN {IQR:,.0f}')
print()
print(f'Lower bound: NGN {lower_bound:,.0f}')
print(f'Upper bound: NGN {upper_bound:,.0f}')

In [ ]:
# Step 2: Flag outliers
outliers = df[(df['salary_monthly'] < lower_bound) | (df['salary_monthly'] > upper_bound)]

print(f'Total rows: {len(df)}')
print(f'Outliers detected: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')
print()
print('Outlier details:')
print(outliers[['name', 'sector', 'role', 'city', 'years_experience', 'salary_monthly']].to_string(index=False))

In [ ]:
# Step 3: Investigate — do the roles and sectors justify these salaries?
print('Outlier sector breakdown:')
print(outliers['sector'].value_counts())
print()
print('Max salary in each sector (all data):')
print(df.groupby('sector')['salary_monthly'].max().sort_values(ascending=False).apply(lambda x: f'NGN {x:,.0f}'))

---
## Part 3 — Z-Score Method (Alternative)

In [ ]:
# Z-score: how many standard deviations from the mean?
mean_s = df['salary_monthly'].mean()
std_s  = df['salary_monthly'].std()

df['z_score'] = (df['salary_monthly'] - mean_s) / std_s

outliers_z = df[df['z_score'].abs() > 3]
print(f'Outliers by z-score (|z| > 3): {len(outliers_z)}')
print()
print(outliers_z[['name', 'sector', 'role', 'salary_monthly', 'z_score']].to_string(index=False))

In [ ]:
# Compare the two methods
iqr_ids = set(outliers.index)
z_ids   = set(outliers_z.index)

print(f'IQR method found:     {len(iqr_ids)} outliers')
print(f'Z-score method found: {len(z_ids)} outliers')
print(f'Found by both:        {len(iqr_ids & z_ids)}')
print(f'Found by IQR only:    {len(iqr_ids - z_ids)}')
print(f'Found by z-score only:{len(z_ids - iqr_ids)}')

---
## Part 4 — Skewness Analysis

In [ ]:
skew_val = df['salary_monthly'].skew()

print(f'Skewness of salary_monthly: {skew_val:.3f}')
print()

if skew_val > 1:
    print('>>> Highly right-skewed (positive skew)')
    print('>>> A small number of very high earners are creating a long right tail.')
    print('>>> Use MEDIAN as the measure of centre — not mean.')
elif skew_val > 0.5:
    print('>>> Moderately right-skewed')
    print('>>> Median is still preferred over mean.')
elif skew_val > -0.5:
    print('>>> Approximately symmetric — mean and median are both valid.')
else:
    print('>>> Left-skewed — use median.')

In [ ]:
# Demonstrate the skew visually with decile analysis
print('Salary by decile (every 10th percentile):')
print()
for p in range(0, 101, 10):
    val = df['salary_monthly'].quantile(p/100)
    bar = '#' * int(val / 50000)
    print(f'  {p:3d}th %ile: NGN {val:>10,.0f}  {bar}')
print()
print('Notice: the jump from 90th to 100th is much larger than from 0th to 10th.')
print('This is the right tail at work.')

---
## Part 5 — Correlation: Experience vs Salary

In [ ]:
# Question 3: Does salary increase with years of experience?
corr = df['years_experience'].corr(df['salary_monthly'])
print(f'Correlation (years_experience vs salary_monthly): {corr:.3f}')
print()

if abs(corr) >= 0.7:
    strength = 'Strong'
elif abs(corr) >= 0.4:
    strength = 'Moderate'
else:
    strength = 'Weak'

direction = 'positive' if corr > 0 else 'negative'
print(f'>>> {strength} {direction} correlation.')
print('>>> Remember: correlation does not prove causation.')

In [ ]:
# Class imbalance check on sector
print('Sector distribution:')
sector_counts = df['sector'].value_counts()
sector_pct    = df['sector'].value_counts(normalize=True).round(3) * 100
for sector in sector_counts.index:
    print(f'  {sector:<25} {sector_counts[sector]:>4} records  ({sector_pct[sector]:.1f}%)')

---
## Part 6 — Decision and Summary

In [ ]:
# Complete five-step outlier investigation summary
print('===== Salary Outlier Investigation Summary =====')
print()
print(f'Step 1 — IQR bounds')
print(f'         Upper bound: NGN {upper_bound:,.0f}')
print()
print(f'Step 2 — Outliers flagged: {len(outliers)} ({len(outliers)/len(df)*100:.1f}% of records)')
print()
print(f'Step 3 — Skewness: {skew_val:.2f} (right-skewed)')
print()
print(f'Step 4 — Mean:   NGN {df["salary_monthly"].mean():,.0f}')
print(f'         Median: NGN {df["salary_monthly"].median():,.0f}')
print()
print(f'Step 5 — Decision: KEEP outliers (plausible senior roles in high-paying sectors)')
print(f'         Recommended summary: MEDIAN')
print()
print(f'Typical Nigerian professional monthly salary: NGN {df["salary_monthly"].median():,.0f}')

---
## Summary

In this demo we:
- Spotted mean/median divergence as the first outlier signal
- Applied the **IQR method** to formally flag outliers
- Applied the **z-score method** and compared results
- Measured **skewness** and interpreted the right-skewed distribution
- Checked **class imbalance** across sectors
- Measured **correlation** between experience and salary
- Made a **documented decision**: keep outliers, use median as summary statistic

**Next — Topic 3:** We start visualising these patterns with Matplotlib — turning numbers into charts.